# Market Sentiment Predictor

Geovanni Jones, Josh Previte, Casey Barrasso

The goal of this project is to predict the directional price movement of a specific stock using both historical price data and news information. Two separate models will predict price movements based on historical data and analyze news sentiment.

In [33]:
import pandas as pd

# Data Ingestion

There are two main datasets to this project. The first is a dataset containing news article content including titles, content, and sentiment scores from various news sources around the AAPL stock. The second dataset (introduced later in the notebook) contains a time series of AAPL stock prices.   

## Sentiment Dataset

In [34]:
#Github didn't allow us to upload a file more than 25MB. This is combining all files into one file
raw_data_1 = pd.read_csv('../data/apple_news_data_1.csv', encoding='latin-1')
raw_data_2 = pd.read_csv('../data/apple_news_data_2.csv', encoding='latin-1')
raw_data_3 = pd.read_csv('../data/apple_news_data_3.csv', encoding='latin-1')
raw_data_4 = pd.read_csv('../data/apple_news_data_4.csv', encoding='latin-1')
raw_data_5 = pd.read_csv('../data/apple_news_data_5.csv', encoding='latin-1')

raw_data = pd.concat([raw_data_1, raw_data_2, raw_data_3, raw_data_4, raw_data_5])[['date', 'title', 'content', 'link', 'symbols', 'tags', 'sentiment_polarity', 'sentiment_neg', 'sentiment_neu', 'sentiment_pos']]

In [35]:
#view a small subsection of the data to ensure formatting looks correct
raw_data.head(10)

,date,title,content,link,symbols,tags,sentiment_polarity,sentiment_neg,sentiment_neu,sentiment_pos
0,2021-04-30T21:38:00+00:00,Techâs Big Five Had Fantastic Pandemics. Her...,"Alphabet, Amazon.com, Apple, Facebook, and Mic...",https://finance.yahoo.com/m/bbe3f7c9-c7af-3c9f...,"AAPL.US, AAPL34.SA, AMZN.US, FB.US, GOOGL.US","AMAZON, APPLE, FACEBOOK, HARLEY FINKELSTEIN, M...",0.395,0,0.871,0.129
1,2021-04-30T20:41:52+00:00,Valuation Remains a Concern for Churchill Capi...,Churchill Capital IV (NYSE:CCIV) stock has gra...,https://finance.yahoo.com/news/valuation-remai...,"AAPL.US, CCIV-UN.US, CCIV.US","BUSINESS COMBINATION, CCIV, CHURCHILL CAPITAL ...",0.998,0.023,0.85,0.127
2,2021-04-30T20:18:00+00:00,Big techâs trillion-dollar pandemic year may...,This week concludes the first quarter earnings...,https://finance.yahoo.com/m/6e441d74-37c1-3642...,"AAPL.US, AAPL34.SA, AMZN.US, MSFT.US","AMAZON, APPLE INC, BIG TECH, MARKET CAPITALIZA...",0,0,1,0
3,2021-04-30T19:32:30+00:00,What Appleâs blowout earnings means for Berk...,"Krish Sankar, Managing Dir. & Cowen Senior Res...",https://finance.yahoo.com/video/apple-blowout-...,"AAPL.US, AAPL34.SA, BRK-A.US, BRK-B.US, DJI.IN...","ALEXIS CHRISTOFOROUS, BERKSHIRE HATHAWAY, COWE...",0,0,1,0
4,2021-04-30T18:49:00+00:00,EU Agrees With Spotify: Apple Is Abusing Its A...,The EU's European Commission just released pre...,https://finance.yahoo.com/m/07524444-63a7-34aa...,"AAPL.US, AAPL34.SA, SPOT.US","APP STORE, APPLE, EUROPEAN COMMISSION, MUSIC S...",-0.71,0.093,0.885,0.022
5,2021-04-30T18:30:06+00:00,ETFs to Stack as Reopening Poses No Hindrance ...,The tech-heavyÂ Nasdaq Composite Index was a s...,https://finance.yahoo.com/news/etfs-stack-reop...,"AAPL.US, AAPL34.SA, COMP.US, FB.US, GOOGL.US, ...","APPLE, CONSENSUS ESTIMATE, EARNINGS PER SHARE,...",0.999,0.027,0.826,0.148
6,2021-04-30T18:19:10+00:00,Big Tech Stocks Flash Red Flag to Regulators A...,"If âBig Techâ was a gamefish, like Faceboo...",https://finance.yahoo.com/news/big-tech-stocks...,"AAPL.US, CRM.US, FB.US, GME.US, GOOGL.US, MSFT...","BITCOIN, DIGITAL TRANSFORMATION, FACEBOOK, MIC...",0.999,0.026,0.861,0.113
7,2021-04-30T18:13:06+00:00,"Top Stock Reports for Apple, Alphabet &amp; Bl...","Friday, April 30, 2021\n\nThe Zacks Research D...",https://finance.yahoo.com/news/top-stock-repor...,"AAPL.US, AAPL34.SA, BLK.US, F.US, GOOGL.US, ZT...","ALPHABET, APPLE, APPLE WATCH, BLACKROCK, RESEA...",0.999,0.042,0.775,0.183
8,2021-04-30T17:54:31+00:00,Why Stem Stock Looks Very Attractive Following...,"Not infrequently, Wall Street focuses excessiv...",https://finance.yahoo.com/news/why-stem-stock-...,"AAPL.US, AMZN.US, BX.US, FSLR.US, GE.US, GEOO3...","CLEAN-ENERGY, STEM STOCK",0.999,0.015,0.83,0.154
9,2021-04-30T17:34:59+00:00,âTechnology is the largest single growth tre...,"Keith Fitz-Gerald, Fitz-Gerald Group Chief Inv...",https://finance.yahoo.com/video/technology-lar...,"AAPL.US, AAPL34.SA, AMZN.US, DJI.INDX, FB.US, ...","ALEXIS CHRISTOFOROUS, KEITH FITZ-GERALD, SIBIL...",0,0,1,0


In [36]:
#check how many NA values there are. We care most about sentiment, date, and content columns
raw_data.isna().sum().sort_values(ascending=False)

tags                  17790
sentiment_pos          5853
sentiment_neu          5812
sentiment_neg          5767
sentiment_polarity     5721
symbols                5539
link                   5340
content                4901
title                  4255
date                   2808
dtype: int64

In [51]:
#force sentiment scores to be a numerical value, NA if not 
for col in ['sentiment_pos','sentiment_neu','sentiment_neg','sentiment_polarity']:
    raw_data[col] = pd.to_numeric(raw_data[col],errors='coerce')

#convert dates to a datetime object and content to a string
raw_data['date'] = pd.to_datetime(raw_data['date'],errors='coerce',utc=True)
raw_data['date'] = raw_data['date'].dt.tz_convert(None)
raw_data['content'] = raw_data['content'].astype('str')

In [52]:
#this data is messy, drop the columns that don't have the necessary data
raw_data = raw_data.dropna(subset=['sentiment_pos','sentiment_neu','sentiment_neg','sentiment_polarity','content','date'])

In [53]:
#Create a single sentiment score that we can use to determine if an article contains negative or positive sentiment
raw_data['Overall Sentiment'] = raw_data['sentiment_pos'] - raw_data['sentiment_neg']
#we can also make this a classification model by coverting to 'Positive', 'Neutral', or 'Negative' here.

The "content" column will be the predictor variable in the first model, and the string will be transformed into a numerical matrix using feature engineering methods such as TF-IDF. The "overall sentiment" variable will be the target variable. The goal of the first model will be to predict the sentiment of a news article given its content.

## Price Timeseries Dataset

In [54]:
ts_data = pd.read_csv('../data/AAPL Stock Data.csv')

In [55]:
ts_data.head(10)

,Dates,AAPL Share Price
0,Jan-03-2005,1.13
1,Jan-04-2005,1.14
2,Jan-05-2005,1.15
3,Jan-06-2005,1.15
4,Jan-07-2005,1.24
5,Jan-10-2005,1.23
6,Jan-11-2005,1.15
7,Jan-12-2005,1.17
8,Jan-13-2005,1.25
9,Jan-14-2005,1.25


In [56]:
#check how many NA values there are
ts_data.isna().sum().sort_values(ascending=False)

Dates               0
AAPL Share Price    0
dtype: int64

In [57]:
#Convert values to the expected formatting
ts_data['Dates'] = pd.to_datetime(ts_data['Dates'])
ts_data['AAPL Share Price'] = ts_data['AAPL Share Price'].astype('float')

## Combining Datasets

In [58]:
#Get the average sentiment by day, makes it possible to join on AAPL daily stock prices
sentiment_data = raw_data.groupby(['date'])[['Overall Sentiment']].mean()

In [61]:
#join on date
all_data = pd.merge(sentiment_data, ts_data, how = 'right', left_on = 'date', right_on = 'Dates')

In [62]:
all_data

,Overall Sentiment,Dates,AAPL Share Price
0,NaN,2005-01-03,1.13
1,NaN,2005-01-04,1.14
2,NaN,2005-01-05,1.15
3,NaN,2005-01-06,1.15
4,NaN,2005-01-07,1.24
...,...,...,...
5293,NaN,2026-01-16,255.53
5294,NaN,2026-01-20,246.70
5295,NaN,2026-01-21,247.65
5296,NaN,2026-01-22,248.35


We have a greater amount of stock price data than news sentiment data on the AAPL stock. We can remove dates without any sentiment analysis at a future date. For right now, this data could be helpful for a traditional forecasting model

In [65]:
all_data['Overall Sentiment'].describe()

count    331.000000
mean       0.096082
std        0.108209
min       -0.420000
25%        0.027375
50%        0.097667
75%        0.157167
max        0.489000
Name: Overall Sentiment, dtype: float64

The overall sentiment score looks to be performing as expected. Scores range from -.42 to .49, but most are positive (which is expected due to Apple's stellar performance during this time period.

The main model is a time series forecasting model, so it will use stock price as the main target variable. There will be two possible forecasting models, the traditional will exclusively use previous stock prices as predictor variables, while the second model will use overall sentiment as another predictor variable.